# Previsão em `future_unseen_examples.csv` (item 2.3) — versão Python

Versão Python de [`analysis/02-3_previsao_futuro.qmd`](../analysis/02-3_previsao_futuro.qmd). Aplico o modelo escolhido (XGBoost, item 2.1) aos 100 imóveis sem preço.

In [1]:
import pandas as pd
import numpy as np
import joblib

DATA = "../data"
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

modelo_xgb = joblib.load("models/modelo_xgb.joblib")
demograf = pd.read_csv(f"{DATA}/zipcode_demographics.csv")
futuro_raw = pd.read_csv(f"{DATA}/future_unseen_examples.csv")

futuro_raw.shape

(100, 18)

## 1. Ano de referência para `idade_casa`

`future_unseen_examples.csv` não tem coluna de data. Preciso de um ano de referência pra calcular `idade_casa`, e a escolha não é neutra: usar o ano corrente empurraria a idade de todos os imóveis pra fora da faixa que o modelo aprendeu no treino (só viu vendas de 2014–2015), arriscando extrapolação. Uso o último ano de venda do treino como referência — mesma decisão da versão R.

In [2]:
casas_treino = pd.read_csv(f"{DATA}/kc_house_data.csv")
casas_treino["date"] = pd.to_datetime(casas_treino["date"], format="%Y%m%dT%H%M%S")
casas_treino = casas_treino[casas_treino["bedrooms"] < 30].copy()

ano_referencia = casas_treino["date"].dt.year.max()
print("ano de referência:", ano_referencia)

ano de referência: 2015


## 2. Feature engineering e previsão

In [3]:
futuro = futuro_raw.copy()
futuro["tem_porao"] = (futuro["sqft_basement"] > 0).astype(int)
futuro["idade_casa"] = ano_referencia - futuro["yr_built"]
futuro["reformado"] = (futuro["yr_renovated"] > 0).astype(int)
futuro = futuro.merge(demograf, on="zipcode", how="left")

vars_modelo = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "grade", "tem_porao", "idade_casa", "reformado",
    "lat", "long", "sqft_living15", "sqft_lot15",
    "medn_hshld_incm_amt", "medn_incm_per_prsn_amt", "hous_val_amt", "per_bchlr", "per_prfsnl"]

pred_log_price = modelo_xgb.predict(futuro[vars_modelo])

resultado = futuro_raw.copy()
resultado.insert(0, "preco_previsto", np.exp(pred_log_price))
resultado.head(10)[["preco_previsto", "bedrooms", "bathrooms", "sqft_living", "grade", "zipcode"]]

,preco_previsto,bedrooms,bathrooms,sqft_living,grade,zipcode
0,"335,666.41",4,1.00,1680,6,98118
1,"607,650.50",3,2.50,2220,8,98115
2,"286,180.16",3,2.25,1630,8,98030
3,"576,535.62",5,2.50,1710,8,98005
4,"243,690.91",2,1.00,850,6,98126
5,"600,066.75",4,2.75,2630,8,98028
6,"308,791.62",4,1.75,2290,7,98178
7,"450,551.88",4,2.00,1730,8,98125
8,"1,462,081.38",3,1.75,2190,9,98112
9,"734,325.88",3,1.75,1850,6,98074


## 3. As previsões fazem sentido?

In [4]:
comparacao = pd.DataFrame({
    "treino (real)": casas_treino["price"].describe(),
    "future_unseen (previsto)": resultado["preco_previsto"].describe(),
})
comparacao

,treino (real),future_unseen (previsto)
count,"21,612.00",100.00
mean,"540,083.52","492,125.84"
std,"367,135.06","281,860.81"
min,"75,000.00","138,476.39"
25%,"321,837.50","290,660.94"
50%,"450,000.00","431,094.64"
75%,"645,000.00","609,321.92"
max,"7,700,000.00","2,090,709.12"


A distribuição das previsões fica dentro da faixa de preços observada no treino, com média e mediana em ordem de grandeza compatível — sem sinal de previsão absurda (negativa, ou uma casa pequena avaliada em milhões).

## 4. Salvando o resultado final

In [5]:
import os
os.makedirs("output", exist_ok=True)
resultado.to_csv("output/previsoes_future_unseen.csv", index=False)
print(f"salvo: output/previsoes_future_unseen.csv ({len(resultado)} linhas)")

salvo: output/previsoes_future_unseen.csv (100 linhas)
